In [13]:
import numpy as np
import csv
import time
from pathlib import Path

# ---------- Project paths ----------
PROJECT_DIR = Path("/Users/apple/Downloads/core4-finance-imc")
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_path = PROJECT_DIR / "core4_mc_dataset_10000.csv"
script_path = DATA_DIR / "core4_generate_mc_dataset.py"


# ---------- Core Monte Carlo pricer ----------
def monte_carlo_call_price(S, K, T, sigma, r, num_paths=100_000, seed=42):
    rng = np.random.default_rng(seed)

    Z = rng.standard_normal(num_paths)

    ST = S * np.exp(
        (r - 0.5 * sigma**2) * T
        + sigma * np.sqrt(T) * Z
    )

    payoffs = np.maximum(ST - K, 0.0)

    return float(
        np.exp(-r * T) * payoffs.mean()
    )


# ---------- Fast batched dataset generator ----------
def generate_mc_dataset(
    output_csv,
    n_scenarios=10_000,
    num_paths=20_000,
    scenario_seed=20260909,
    mc_seed=12345,
    batch_size=100,
):
    """
    Generate synthetic European call-option scenarios.

    Sampling:
        K ~ Uniform(50, 200)
        moneyness = S/K ~ Uniform(0.7, 1.3)
        S = K * moneyness
        T ~ Uniform(0.05, 2.0)
        sigma ~ Uniform(0.10, 0.60)
        r ~ Uniform(0.00, 0.08)
    """

    scenario_rng = np.random.default_rng(scenario_seed)
    mc_rng = np.random.default_rng(mc_seed)

    # Generate scenario parameters
    K_all = scenario_rng.uniform(50.0, 200.0, n_scenarios)
    m_all = scenario_rng.uniform(0.7, 1.3, n_scenarios)

    S_all = K_all * m_all

    T_all = scenario_rng.uniform(0.05, 2.0, n_scenarios)
    sigma_all = scenario_rng.uniform(0.10, 0.60, n_scenarios)
    r_all = scenario_rng.uniform(0.00, 0.08, n_scenarios)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    start = time.perf_counter()
    rows_written = 0

    with output_csv.open("w", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "scenario_id",
            "S",
            "K",
            "T",
            "sigma",
            "r",
            "moneyness",
            "num_paths",
            "mc_runtime_ms",
            "mc_price",
        ])

        for start_idx in range(0, n_scenarios, batch_size):

            end_idx = min(
                start_idx + batch_size,
                n_scenarios
            )

            b = end_idx - start_idx

            S = S_all[start_idx:end_idx]
            K = K_all[start_idx:end_idx]
            T = T_all[start_idx:end_idx]
            sigma = sigma_all[start_idx:end_idx]
            r = r_all[start_idx:end_idx]
            m = m_all[start_idx:end_idx]

            batch_t0 = time.perf_counter()

            # Each row = one option scenario
            # Each column = one Monte Carlo path
            Z = mc_rng.standard_normal(
                (b, num_paths)
            )

            ST = S[:, None] * np.exp(
                (
                    r[:, None]
                    - 0.5 * sigma[:, None]**2
                )
                * T[:, None]
                + sigma[:, None]
                * np.sqrt(T[:, None])
                * Z
            )

            payoffs = np.maximum(
                ST - K[:, None],
                0.0
            )

            prices = (
                np.exp(-r * T)
                * payoffs.mean(axis=1)
            )

            batch_ms = (
                time.perf_counter()
                - batch_t0
            ) * 1000

            approx_per_scenario_ms = (
                batch_ms / b
            )

            for j in range(b):

                idx = start_idx + j

                writer.writerow([
                    idx + 1,
                    round(float(S[j]), 6),
                    round(float(K[j]), 6),
                    round(float(T[j]), 6),
                    round(float(sigma[j]), 6),
                    round(float(r[j]), 6),
                    round(float(m[j]), 6),
                    num_paths,
                    round(
                        float(approx_per_scenario_ms),
                        6
                    ),
                    round(float(prices[j]), 6),
                ])

            rows_written += b

    total_sec = time.perf_counter() - start

    return {
        "file": str(output_csv),
        "rows": rows_written,
        "num_paths_per_scenario": num_paths,
        "runtime_seconds": total_sec,
    }


# ---------- Generate dataset ----------
summary = generate_mc_dataset(
    csv_path,
    n_scenarios=10_000,
    num_paths=20_000,
    batch_size=100,
)


# ---------- Test one normal option ----------
test_price = monte_carlo_call_price(
    S=100,
    K=100,
    T=1,
    sigma=0.20,
    r=0.05,
    num_paths=200_000,
    seed=42,
)


print(summary)
print("Test price:", test_price)
print("CSV saved to:", csv_path)

{'file': '/Users/apple/Downloads/core4-finance-imc/core4_mc_dataset_10000.csv', 'rows': 10000, 'num_paths_per_scenario': 20000, 'runtime_seconds': 4.918659547984134}
Test price: 10.463413536194789
CSV saved to: /Users/apple/Downloads/core4-finance-imc/core4_mc_dataset_10000.csv
